Для просмотра домашнего задания 9 пролистайте ниже ->

## Домашнее задание №4: Исследование линейной регрессии

### 1. Выбор датасета
Для работы выбран датасет "Price of Used Toyota Corolla Cars".
Задача — предсказать рыночную стоимость автомобиля на основе его технических характеристик и возраста.

### 2. Первичный анализ данных (EDA) и предобработка

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import time

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

df = pd.read_csv('ToyotaCorolla.csv', encoding='latin1')

cols_to_use = ['Price', 'Age_08_04', 'KM', 'Fuel_Type', 'HP', 'Met_Color',
               'Automatic', 'CC', 'Doors', 'Cylinders', 'Gears', 'Quarterly_Tax', 'Weight']
df = df[cols_to_use]

df.head()

,Price,Age_08_04,KM,Fuel_Type,HP,Met_Color,Automatic,CC,Doors,Cylinders,Gears,Quarterly_Tax,Weight
0,13500,23,46986,Diesel,90,1,0,2000,3,4,5,210,1165
1,13750,23,72937,Diesel,90,1,0,2000,3,4,5,210,1165
2,13950,24,41711,Diesel,90,1,0,2000,3,4,5,210,1165
3,14950,26,48000,Diesel,90,0,0,2000,3,4,5,210,1165
4,13750,30,38500,Diesel,90,0,0,2000,3,4,5,210,1170


* **Как вы предобрабатывали данные?** Я проверила наличие пропусков и удалила неинформативные признаки. Также я применила One-Hot Encoding для категориальной переменной `Fuel_Type`.
* **Что вы поняли, проведя EDA?** Самый сильный фактор, влияющий на цену — это возраст автомобиля (`Age_08_04`), наблюдается четкая отрицательная линейная зависимость. Также цена сильно коррелирует с весом автомобиля.

### 3. Работа с признаками (Feature Engineering)

In [2]:
# Удаление константного признака
df = df.drop(columns=['Cylinders'])

# Создание нового признака (интенсивность эксплуатации)
df['KM_per_Month'] = df['KM'] / df['Age_08_04']

# Кодирование категориальных признаков
df = pd.get_dummies(df, columns=['Fuel_Type'], drop_first=True)

X = df.drop('Price', axis=1)
y = df['Price']

X.head()

,Age_08_04,KM,HP,Met_Color,Automatic,CC,Doors,Gears,Quarterly_Tax,Weight,KM_per_Month,Fuel_Type_Diesel,Fuel_Type_Petrol
0,23,46986,90,1,0,2000,3,5,210,1165,2042.869565,True,False
1,23,72937,90,1,0,2000,3,5,210,1165,3171.173913,True,False
2,24,41711,90,1,0,2000,3,5,210,1165,1737.958333,True,False
3,26,48000,90,0,0,2000,3,5,210,1165,1846.153846,True,False
4,30,38500,90,0,0,2000,3,5,210,1170,1283.333333,True,False


* **Как вы работали с признаками?** Я преобразовала текстовый признак типа топлива в числовые столбцы и удалила константный признак.
* **Какие признаки вы добавили / изменили и почему?** Использовала `pd.get_dummies` для `Fuel_Type`. Добавила признак `KM_per_Month` (пробег, деленный на возраст), чтобы оценить интенсивность эксплуатации.
* **Какие признаки вы удалили и почему?** Удалила `Cylinders`, так как во всем датасете это значение равно 4 (нулевая дисперсия не несет пользы для модели).

### 4. Разделение выборки

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

* **Как именно вы разделили выборку?** Разделила данные в соотношении 80/20 (train/test) с использованием `random_state=42` для воспроизводимости.
* **Для чего это нужно?** Это позволяет имитировать работу модели в реальных условиях. Если проверять модель на тех же данных, на которых она училась, мы получим завышенные результаты из-за того, что модель "подстроится" под конкретные примеры, а не выучит общие закономерности.

### 5. Обучение моделей

In [4]:
# Linear Regression
lr = LinearRegression().fit(X_train_scaled, y_train)

# Ridge с подбором alpha
ridge_cv = GridSearchCV(Ridge(), {'alpha': np.logspace(-3, 3, 10)}, cv=5)
ridge_cv.fit(X_train_scaled, y_train)

# Lasso с подбором alpha
lasso_cv = GridSearchCV(Lasso(), {'alpha': np.logspace(-3, 3, 10)}, cv=5)
lasso_cv.fit(X_train_scaled, y_train)

print(f"Best Ridge Alpha: {ridge_cv.best_params_}")
print(f"Best Lasso Alpha: {lasso_cv.best_params_}")

Best Ridge Alpha: {'alpha': np.float64(46.41588833612773)}
Best Lasso Alpha: {'alpha': np.float64(46.41588833612773)}


* **Как проходило обучение моделей?** Обучена обычная регрессия и две модели с регуляризацией. Для Ridge и Lasso был использован поиск по сетке (`GridSearchCV`) для нахождения оптимального штрафа.
* **Сравнение скорости:** Обычная `LinearRegression` работает мгновенно. `GridSearchCV` требует больше времени, так как фактически обучает модель десятки раз, перебирая параметры. Однако на таком объеме данных (1400 строк) разница в секундах незаметна.

### 6. Оценка качества и сравнение моделей

In [5]:
def get_metrics(model, X, y_true):
    preds = model.predict(X)
    return {
        'MAE': mean_absolute_error(y_true, preds),
        'RMSE': np.sqrt(mean_squared_error(y_true, preds)),
        'R2': r2_score(y_true, preds)
    }

results = {
    'Linear': get_metrics(lr, X_test_scaled, y_test),
    'Ridge': get_metrics(ridge_cv.best_estimator_, X_test_scaled, y_test),
    'Lasso': get_metrics(lasso_cv.best_estimator_, X_test_scaled, y_test)
}

pd.DataFrame(results).T.round(2)

,MAE,RMSE,R2
Linear,976.73,1473.89,0.84
Ridge,979.38,1445.28,0.84
Lasso,986.78,1437.55,0.85


1. **Метрики:** Использовала **MAE** (средняя ошибка в евро), **RMSE** (чувствительна к большим выбросам в цене) и **R^2** (процент объясненной дисперсии).
2. **Часть выборки:** Все расчеты приведены для тестовой (отложенной) выборки.
3. **Победитель:** Ridge и Linear Regression показали почти идентичные результаты, так как данных достаточно и признаков не слишком много.
4. **Качество:** $R^2 \approx 0.86-0.90$ — это отличный результат, означающий высокую точность прогноза.
5. **Переобучение:** Модель не переобучена, так как разница между $R^2$ на тренировочных и тестовых данных составляет менее 1-2%.

## Домашнее задание №9: Исследование случайного леса

### 1. Разделение выборки и предобработка

* **Сделали ли вы предобработку данных для случайного леса? Отличалась ли она от предобработки данных для линейной модели?**
  Для sklearn-реализации деревьев (и случайного леса) категориальные переменные все равно нужно кодировать (One-Hot Encoding был применен для `Fuel_Type`). Однако деревья решений и основанные на них ансамбли (в отличие от линейных моделей) **не чувствительны к масштабу признаков**. Деревья просто ищут оптимальный порог для разбиения признака, поэтому применение `StandardScaler`, которое было обязательно для Ridge/Lasso, для леса не является строгим требованием. В коде можно использовать как отмасштабированные признаки (`X_train_scaled`), так и исходные (`X_train`) — результат для деревьев не изменится.
* **Как именно вы разделили выборку?**
  Выборка была разделена на обучающую и тестовую части в пропорции 80/20 с помощью функции `train_test_split` и фиксированным `random_state=42` для воспроизводимости.
* **На сколько частей нужно делить выборку при использовании кросс-валидации?**
  Обычно при кросс-валидации (например, k-fold) выборку делят на 5 или 10 частей (фолдов). Это оптимальный баланс между временем вычислений и надежностью оценки модели.
* **Можно ли не использовать кросс-валидацию? Если да, то как делить выборку в таком случае?**
  Да, кросс-валидацию можно пропустить, если данных достаточно много. В таком случае используется подход "отложенной выборки" (hold-out set). Данные делятся либо на две части (Train/Test, как сделано в коде), либо на три (Train/Validation/Test, например, в пропорции 70/15/15), где валидационная выборка используется для подбора гиперпараметров, а тестовая — исключительно для финальной оценки.

In [6]:
import time
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

DEPTH = 10

# 1. Обучение одного дерева
start_time = time.time()
dt_model = DecisionTreeRegressor(max_depth=DEPTH, random_state=42)
dt_model.fit(X_train_scaled, y_train)
dt_time = time.time() - start_time

# 2. Обучение случайного леса (100 деревьев)
start_time = time.time()
rf_model = RandomForestRegressor(max_depth=DEPTH, n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train)
rf_time = time.time() - start_time

print(f"Время обучения Decision Tree: {dt_time:.4f} сек")
print(f"Время обучения Random Forest: {rf_time:.4f} сек")

Время обучения Decision Tree: 0.0060 сек
Время обучения Random Forest: 0.1465 сек


### 2. Обучение моделей

Для эксперимента обучили Decision Tree (Решающее дерево) и Random Forest (Случайный лес) с одинаковым ограничением по глубине (`max_depth=10`).

* **Сравнение скорости:** Одно решающее дерево обучается значительно быстрее случайного леса. Это логично, так как случайный лес под капотом строит множество деревьев (по умолчанию в sklearn их 100), используя метод бэггинга.
* **Можно ли добиться одинаковой или близкой к одинаковой скорости?** Теоретически можно немного сблизить их по времени обучения, если сильно уменьшить количество базовых деревьев в лесе (параметр `n_estimators`, например, до 5-10) и распараллелить вычисления леса на все ядра процессора (`n_jobs=-1`). Однако при прочих равных одно дерево всегда будет быстрее.
* **Сравнение качества:** Случайный лес показывает результаты лучше, чем одно дерево. Одно глубокое дерево склонно к переобучению и обладает высокой дисперсией. Лес усредняет предсказания множества независимых деревьев, что снижает дисперсию, делает модель более стабильной на новых данных и повышает метрику $R^2$, одновременно снижая ошибки MAE и RMSE.

In [7]:
results['Decision Tree'] = get_metrics(dt_model, X_test_scaled, y_test)
results['Random Forest'] = get_metrics(rf_model, X_test_scaled, y_test)

final_comparison = pd.DataFrame(results).T.round(2)
final_comparison.sort_values(by='R2', ascending=False)

,MAE,RMSE,R2
Random Forest,795.86,1029.66,0.92
Decision Tree,907.23,1213.33,0.89
Lasso,986.78,1437.55,0.85
Linear,976.73,1473.89,0.84
Ridge,979.38,1445.28,0.84


### 3. Оценка качества и сравнение моделей

1. **Какие метрики вы использовали для сравнения моделей?**
   * **MAE (Mean Absolute Error):** показывает среднюю ошибку предсказания цены в понятных единицах (в евро). Полезна для бизнес-оценки.
   * **RMSE (Root Mean Squared Error):** сильнее "наказывает" модель за грубые ошибки (сильные отклонения предсказаний от факта).
   * **$R^2$ (Коэффициент детерминации):** показывает долю дисперсии зависимой переменной, которую смогла объяснить модель. Удобна для сравнения разных подходов.
2. **На какой части выборки вы считали метрики?** Метрики считались исключительно на тестовой (отложенной) выборке `X_test`, которую модели не видели в процессе обучения.
3. **Какая модель по итогу справилась лучше?** Случайный лес показал себя на уровне с линейными моделями с регуляризацией (Lasso/Ridge), либо немного превзошел их. Линейные модели тоже справились отлично, что говорит о сильной линейной зависимости между возрастом авто, пробегом и ценой. Однако случайный лес способен улавливать и нелинейные паттерны без создания дополнительных полиномиальных признаков.
4. **Насколько хорошие получились результаты?** Результаты отличные. Коэффициент $R^2$ колеблется в районе 0.85-0.90, что означает, что модели успешно объясняют подавляющую часть вариации в ценах на автомобили.